# Visualize Effect

This notebook shows what the imposed effects look like on a sample 2d image

In [ ]:
import hglm 
import nibabel as nib
import numpy as np

# build mean FA from HCP data
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp_hcp = hglm.experiment.ExperimentImageOnly.from_search(
    folder=folder,
    sbj_regex=r'[\d]{6}',
    img_glob_dict={'FA': '*_FA.nii.gz'})

exp = exp_hcp.sample_x(a=2, seed=1)

# build image of average y
y_mean = exp.y.mean(axis=1)
y_mean_img = np.zeros(exp.mask_idx.shape)
y_mean_img[exp.mask_idx > -1] = y_mean.flatten()

# get horizontal image
middle = tuple(_xyz // 2 for _xyz in y_mean_img.shape)
y = y_mean_img[:, :, middle[2]]

# trim and cast to mask_idx form
mask_idx = hglm.mask.get_mask_idx(y)
y = y[mask_idx > -1].reshape(1, 1, -1)
mask_idx = hglm.mask.trim_zeros_2d(mask_idx, to_trim=-1)

In [ ]:
num_img = 3
noise_scale = .5

# build experiment
exp_slice = hglm.experiment.ExperimentImageOnly(y=y, mask_idx=mask_idx)
exp_slice.bootstrap_img(n=num_img, noise_scale=noise_scale)
b = exp_slice.y.shape[0]
exp_slice = hglm.experiment.Experiment(x=np.arange(num_img).reshape(1, -1), 
                                       contrast=np.ones(b, dtype=bool),
                                       y=exp_slice.y,
                                       mask_idx=exp_slice.mask_idx)

In [ ]:
import matplotlib.pyplot as plt
from skimage import measure

def plot_slice(y, mask_idx, mask=None, **kwargs):
    # build image
    img = np.full(mask_idx.shape, np.nan)
    img[mask_idx > -1] = y
    plt.imshow(img, cmap='viridis', **kwargs)
    plt.axis('off')
    
    if mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            plt.gca().plot(contour[:, 1], contour[:, 0], linewidth=3, color='red')
    
def plot_all(exp, **kwargs):
    fig, ax = plt.subplots(1, num_img)
    fig.set_size_inches((15, 5))
    vmin, vmax = exp.y[0, ...].min(), exp.y[0, ...].max()
    for img_idx in range(num_img):
        plt.sca(ax[img_idx])
        plot_slice(y=exp.y[0, img_idx, :], 
                   mask_idx=exp_slice.mask_idx,
                   vmin=vmin, vmax=vmax,
                   **kwargs)
        ax[img_idx].set_title(f'img{img_idx} (x={exp_slice.x[0, img_idx]})')
    plt.colorbar()
    

In [ ]:
plot_all(exp_slice)

In [ ]:
ext_sphere = hglm.effect.ExtenterSphere(radius=10)

_exp, effect = exp_slice.impose_effect(pval=.05, extenter=ext_sphere, seed=0)
plot_all(_exp, mask=effect.mask)